# 06 Predict A Match

Load the persisted logistic-regression model and predict the result of one future blue-vs-red matchup. The notebook is self-contained: it reads `clean_matches.csv`, the trained model, and the saved feature column list directly from `artifacts/`.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


def resolve_artifacts_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd / "artifacts", cwd.parent / "artifacts"]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return cwd.parent / "artifacts" if cwd.name == "notebooks" else cwd / "artifacts"


ARTIFACTS_DIR = resolve_artifacts_dir()

clean_matches = pd.read_csv(ARTIFACTS_DIR / "clean_matches.csv", parse_dates=["date"])
model = joblib.load(ARTIFACTS_DIR / "logistic_regression_model.joblib")
feature_columns = json.loads((ARTIFACTS_DIR / "feature_columns.json").read_text())
clean_matches.head()

## Feature-engineering helpers

Define the same Elo and rolling-history feature logic as notebook 02 inline, so this notebook does not depend on the `mlops` source module at runtime.

In [ ]:
def expected_score(rating_a: float, rating_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400.0))


def safe_mean(values: list[int], default: float = 0.5) -> float:
    return float(np.mean(values)) if values else default


def coerce_blue_team_win(row) -> int | None:
    if not pd.isna(row.blue_team_win):
        return int(row.blue_team_win)

    if pd.isna(row.winner) or pd.isna(row.blue_team) or pd.isna(row.red_team):
        return None
    if row.winner == row.blue_team:
        return 1
    if row.winner == row.red_team:
        return 0
    return None


def build_match_features_local(
    df: pd.DataFrame,
    base_elo: float = 1500.0,
    k_factor: float = 32.0,
) -> pd.DataFrame:
    ordered = df.sort_values(["date"], kind="mergesort").reset_index(drop=True).copy()
    elo = defaultdict(lambda: base_elo)
    matches_played = defaultdict(int)
    overall_history = defaultdict(list)
    blue_side_history = defaultdict(list)
    red_side_history = defaultdict(list)
    last_played = {}
    head_to_head = defaultdict(list)
    rows = []

    for row in ordered.itertuples(index=False):
        blue = row.blue_team
        red = row.red_team
        match_date = row.date
        key = tuple(sorted((blue, red)))
        blue_h2h = head_to_head[key]

        rows.append(
            {
                "season": row.season,
                "date": match_date,
                "event": row.event,
                "patch": row.patch,
                "blue_team": blue,
                "red_team": red,
                "winner": row.winner,
                "blue_team_win": row.blue_team_win,
                "elo_diff": elo[blue] - elo[red],
                "winrate_last_5_diff": safe_mean(overall_history[blue][-5:]) - safe_mean(overall_history[red][-5:]),
                "winrate_last_10_diff": safe_mean(overall_history[blue][-10:]) - safe_mean(overall_history[red][-10:]),
                "winrate_last_20_diff": safe_mean(overall_history[blue][-20:]) - safe_mean(overall_history[red][-20:]),
                "matches_played_diff": matches_played[blue] - matches_played[red],
                "days_since_last_match_diff": (
                    (match_date - last_played[blue]).days if blue in last_played else -1
                ) - (
                    (match_date - last_played[red]).days if red in last_played else -1
                ),
                "head_to_head_winrate_diff": (
                    sum(1 for winner in blue_h2h if winner == blue) / len(blue_h2h) if blue_h2h else 0.5
                ) - 0.5,
                "blue_side_team_winrate": safe_mean(blue_side_history[blue]),
                "red_side_team_winrate": safe_mean(red_side_history[red]),
            }
        )

        outcome = coerce_blue_team_win(row)
        if outcome is None:
            continue

        expected = expected_score(elo[blue], elo[red])
        elo[blue] += k_factor * (outcome - expected)
        elo[red] += k_factor * ((1 - outcome) - (1 - expected))
        matches_played[blue] += 1
        matches_played[red] += 1
        overall_history[blue].append(outcome)
        overall_history[red].append(1 - outcome)
        blue_side_history[blue].append(outcome)
        red_side_history[red].append(1 - outcome)
        last_played[blue] = match_date
        last_played[red] = match_date
        head_to_head[key].append(blue if outcome == 1 else red)

    return pd.DataFrame(rows)

## Build the upcoming-match feature row

Edit the values below to predict any matchup. Unknown teams fall back to the default Elo rating and empty rolling history, which is fine — the model still produces a calibrated probability.

In [ ]:
def build_upcoming_match_feature_row(
    history_df: pd.DataFrame,
    match_date: str,
    blue_team: str,
    red_team: str,
    event: str,
    patch: str,
    season: int | None,
) -> pd.DataFrame:
    future_match = pd.DataFrame(
        [
            {
                "season": season,
                "date": pd.to_datetime(match_date),
                "event": event,
                "patch": patch,
                "blue_team": blue_team,
                "red_team": red_team,
                "winner": pd.NA,
                "blue_team_win": pd.NA,
            }
        ]
    )
    combined = pd.concat([history_df.copy(), future_match], ignore_index=True, sort=False)
    feature_rows = build_match_features_local(combined)
    return feature_rows.tail(1).reset_index(drop=True)


match_date = "2026-06-01"
blue_team = "Team A"
red_team = "Team B"
event = "Main"
patch = "14.1"
season = 1

match_features = build_upcoming_match_feature_row(
    history_df=clean_matches,
    match_date=match_date,
    blue_team=blue_team,
    red_team=red_team,
    event=event,
    patch=patch,
    season=season,
)
match_features

## Predict and export the result

Select the trained feature columns, run the model, and persist the prediction to `artifacts/predicted_match_result.csv`.

In [ ]:
prediction_frame = match_features[feature_columns]
blue_win_prob = float(model.predict_proba(prediction_frame)[0, 1])
red_win_prob = 1.0 - blue_win_prob
predicted_winner = blue_team if blue_win_prob >= 0.5 else red_team

result_df = pd.DataFrame(
    [
        {
            "date": match_date,
            "blue_team": blue_team,
            "red_team": red_team,
            "blue_win_prob": blue_win_prob,
            "red_win_prob": red_win_prob,
            "predicted_winner": predicted_winner,
        }
    ]
)
result_df.to_csv(ARTIFACTS_DIR / "predicted_match_result.csv", index=False)
result_df